# University of Kentucky Flowsheet Optimization Tutorial 
Throughout this tutorial, you will learn how to **write an optimization function** for a given flowsheet and **iteratively optimize the flowsheet**, while also gaining an understanding of the engineering insights used to justify the system's operating conditions. Our case study will be the University of Kentucky (UKy) flowsheet, a continuous, pilot-scale plant operation for recovering rare earth elements (REEs) from coal and coal byproducts with the goal of producing marketable mixed rare earth oxides (REOs), as shown in the process flow diagram below. For more information on this flowsheet, please refer to the Useful Links at the bottom of this cell. 


## UKy Flowsheet Connectivity
![uky_flowsheet.png](./uky_flowsheet.png)


## Tutorial Outline
In this tutorial, we will start with an outdated version of the UKy flowsheet that is only able to achieve 0.94% REE recovery. Then, we will write an optimization function that unfixes a variety of operating conditions in order to maximize the REE recovery while still adhering to realistic variable bounds. From there, we will walk through an iterative process of running the optimization, updating the flowsheet operating conditions, diagnosing any issues, and then expanding the bounds of the optimization until we hit the target recovery of ~20% REE.


### Useful Links:
- Public Github Repository: https://github.com/prommis/prommis/tree/main
- Tutorial on Building the UKy Flowsheet: [UKy Flowsheet Tutorial](uky_flowsheet-solution.ipynb)
- **Current** UKy Flowsheet Code: https://github.com/prommis/prommis/blob/main/src/prommis/uky/uky_flowsheet.py
- **Outdated** UKy Flowsheet Code: [Outdated UKy](uky_flowsheet_low_recovery.py)


## Step 1: Import the necessary tools

We'll use some basic functionalities from Pyomo, generic models from IDAES, and process-specific models from PrOMMiS.

In [1]:
# Import the essentials from Pyomo
from pyomo.environ import (
    Constraint,
    Objective,
    value,
)
from pyomo.network import SequentialDecomposition

# Import the essentials from IDAES
import idaes.logger as idaeslog

# Import scaling, initialization, and diagnostic tools from IDAES
from idaes.core.initialization import BlockTriangularizationInitializer
from idaes.core.scaling.util import get_scaling_factor, set_scaling_factor
from idaes.core.solvers import get_solver

# Import unit models from IDAES
from idaes.models.unit_models.feed import FeedInitializer
from idaes.models.unit_models.mixer import MixerInitializer
from idaes.models.unit_models.product import ProductInitializer
from idaes.models.unit_models.separator import SeparatorInitializer

# Import the UKy-specific unit and property models
from prommis.leaching.leach_train import LeachingTrainInitializer
from prommis.solvent_extraction.solvent_extraction import SolventExtractionInitializer

# Import parameter sweep tools from WaterTAP
from parameter_sweep import LinearSample, parameter_sweep

# Import the outdated UKy flowsheet
import uky_flowsheet_low_recovery as uky

# Set up logger
_log = idaeslog.getLogger(__name__)

'idaes.co re.util.diagnostics_tools.diagnostics_toolbox.DiagnosticsToolbox'.
Please update your import.  (deprecated in 2.12.0) (called from <frozen
importlib._bootstrap>:241)


## Step 2: Run the outdated UKy Flowsheet
We will start by running an outdated version of the UKy flowsheet that only recovered 0.9% of the total REEs. To do so, we will simply import the functions directly from the flowsheet. If you'd like to take a closer look at these functions, see [Outdated UKy](uky_flowsheet_low_recovery.py).

In [2]:
m1 = uky.build()
uky.set_operating_conditions(m1, DEHPA_dosage=0.05)
uky.set_scaling(m1)
uky.initialize_system(m1)
uky.solve_system(m1, tee=False)
uky.fix_organic_recycle(m1)
uky.solve_system(m1, tee=False)
uky.add_result_expressions(m1)

Scaling fs.leach
Scaling fs.sl_sep1
Scaling fs.scrubber_HCl_leach_translator
Scaling fs.leach_mixer
Scaling fs.leach_liquid_feed
Scaling fs.leach_solid_feed
Scaling fs.leach_filter_cake
Scaling fs.leach_filter_cake_liquid
Scaling fs.rougher_org_make_up
Scaling fs.solex_rougher_load
Scaling fs.acid_feed1
Scaling fs.solex_rougher_scrub
Scaling fs.acid_feed2
Scaling fs.solex_rougher_strip
Scaling fs.rougher_sep
Scaling fs.rougher_mixer
Scaling fs.load_sep
Scaling fs.scrub_sep
Scaling fs.rougher_organic_purge
Scaling fs.solex_cleaner_load
Scaling fs.solex_cleaner_strip
Scaling fs.cleaner_org_make_up
Scaling fs.cleaner_mixer
Scaling fs.cleaner_sep
Scaling fs.cleaner_HCl_leach_translator
Scaling fs.leach_sx_mixer
Scaling fs.acid_feed3
Scaling fs.cleaner_organic_purge
Scaling fs.precipitator
Scaling fs.sl_sep2
Scaling fs.precip_sep
Scaling fs.precip_sx_mixer
Scaling fs.precip_purge
Scaling fs.roaster
Scaling fs.leaching_sol_feed_expanded
Scaling fs.leaching_liq_feed_expanded
Scaling fs.leachi

In [3]:
print(f"Total REE recovery is {value(m1.fs.overall_ree_recovery_percentage[0])} %")

Total REE recovery is 0.9049600258591483 %


## Step 3: Write an optimization function to maximize REE recovery
Now we want to write an optimization function that maximizes the total REE recovery. So we'll need to write an objective function and unfix various operating conditions.

### Step 3.1 Create objective function
In our objective function, we'll want to maximize the REE recovery without using an excessive volume of feed streams and limiting the amount of contaminants in the final product. We can model this by assigning penalties (positive or negative coefficients) to each of these quantities with the goal of minimizing the objective function. Likewise, we'll set an arbitrarily large, negative coefficient for the REE product flow, which is directly correlated to the total REE recovery. Since the purity of the final product is just as important as the mass of REEs recovered, we'll assign an equally large, positive coefficient for the total contaminants in the product flow. Lastly, we'll assign a small penalty for using larger flow rates than necessary; otherwise, the objective function would increase the mass of REE recovered by simply inflating the flow of REEs coming into the system.

In [4]:
def create_objective(m):
    # Goal: Minimize this objective function
    m.obj = Objective(
        expr=(
            # Assign a small penalty for having larger flow rates than necessary
            0.01
            * (
                m.fs.leach_liquid_feed.flow_vol[0]
                + m.fs.acid_feed1.flow_vol[0]
                + m.fs.acid_feed2.flow_vol[0]
                + m.fs.acid_feed3.flow_vol[0]
            )
            # Assign a large penalty for having contaminants
            + 1e5
            * (
                +m.fs.metal_product_flow[0, "Al"]
                + m.fs.metal_product_flow[0, "Ca"]
                + m.fs.metal_product_flow[0, "Fe"]
            )
            # Assign a large incentive for increasing REE product flow (REE recovery)
            - 1e5 * m.fs.ree_product_flow[0]
        )
    )

### Step 3.2 Unfix liquid feed conditions
Now we have an objective function that we can use to optimize the system for REE recovery, but all of the system's variables are still fixed. Thus, the next few steps will involve unfixing various operating conditions (design decisions) that the objective function can change, as it sees fit. To ensure the objective function does not give these variables unrealistic values, we'll assign bounds and/or define constraints so that the optimized solution does not violate any laws of chemistry or explore scenarios outside the bounds of what we think is plasuible.

We'll start by unfixing the flow rate and concentration of the liquid leaching feed. Constraints are added to uphold chemistry, scaling is updated to lighten the burden on the solver, and a lower bound is set for the volumetric flow rate to ensure realistic operating conditions.

In [5]:
def optimize_liquid_feed(m):
    # Unfix the H2SO4 feed rate and feed concentration
    m.fs.leach_liquid_feed.flow_vol.unfix()
    m.fs.leach_liquid_feed.properties[0].flow_vol.setlb(100)

    m.fs.leach_liquid_feed.conc_mass_comp[0, "H"].unfix()
    m.fs.leach_liquid_feed.conc_mass_comp[0, "HSO4"].unfix()
    m.fs.leach_liquid_feed.conc_mass_comp[0, "SO4"].unfix()

    # Ensure that stoichiometry is upheld
    @m.fs.leach_liquid_feed.Constraint(m.fs.time)
    def H2SO4_stoich_eqn(b, t):
        return (
            b.properties[t].conc_mol_comp["H"]
            == 2 * b.properties[t].conc_mol_comp["SO4"]
            + b.properties[t].conc_mol_comp["HSO4"]
        )

    # Apply a scaling factor of 10 to conc_mol_comp values
    for condata in m.fs.leach_liquid_feed.H2SO4_stoich_eqn.values():
        set_scaling_factor(condata, 10)

    # Ensure chemical equilibrium for HSO4 dissociation
    @m.fs.leach_liquid_feed.Constraint(m.fs.time)
    def HSO4_dissociation(b, t):
        return (
            b.properties[t].params.Ka2 * b.properties[t].conc_mol_comp["HSO4"]
            == b.properties[t].conc_mol_comp["SO4"] * b.properties[t].conc_mol_comp["H"]
        )

    # Update scaling factors
    sf = get_scaling_factor(m.fs.leach.mscontactor.liquid[0, 1].hso4_dissociation)
    for condata in m.fs.leach_liquid_feed.HSO4_dissociation.values():
        set_scaling_factor(condata, sf)

<div class="alert alert-block alert-info">
<b>Engineering Insight:</b>
After solving, the objective function originally wanted to use a liquid feed of 13.7 L/hr, which would be far too low to let the leach solids move as a slurry. The original flow rate was 224.3 L/hr, so we use 100 as a rough estimate for a lower bound on the liquid feed flow rate.
</div>

### Step 3.3 Unfix organic feed conditions
Similarly, we'll unfix the flow rates and DEHPA concentration of the organic feed streams. 

In [6]:
def optimize_organic_feed(m, DEHPA_ub=10):
    m.fs.rougher_mixer.outlet.flow_vol.unfix()
    m.fs.cleaner_mixer.outlet.flow_vol.unfix()

    m.fs.rougher_org_make_up.conc_mass_comp[0, "DEHPA"].unfix()
    m.fs.cleaner_org_make_up.conc_mass_comp[0, "DEHPA"].unfix()
    m.fs.rougher_org_make_up.properties[0].extractant_dosage.bounds = (5, DEHPA_ub)
    m.fs.cleaner_org_make_up.properties[0].extractant_dosage.bounds = (5, DEHPA_ub)

<div class="alert alert-block alert-info">
<b>Engineering Insight:</b>
We apply an upper bound of 10 for the DEHPA dosage, where 5 is the default value. From our research, we've inferred that the DEHPA volume-to-volume ratio would realistically be between 20-25%. However, if we try to jump straight from solving the system at 5% to solving the system at 20%, the solve will fail as the solutions are too far apart. Thus, we will only jump to 10% at this stage and increase this upper bound later on in the tutorial after we've updated the initialization routine.
</div>

### Step 3.4 Unfix acid feed conditions
Now we'll unfix the flow rates and concentrations of the acid feed streams, while ensuring the stoichiometry remains consistent.

In [7]:
def optimize_acid_feed(m):
    # Unfix HCl feed flow rates and concentrations
    for feed in [m.fs.acid_feed1, m.fs.acid_feed2, m.fs.acid_feed3]:
        feed.flow_vol.unfix()
        feed.conc_mass_comp[0, "H"].unfix()
        feed.conc_mass_comp[0, "Cl"].unfix()

        feed.properties[0].pH_phase["liquid"].setlb(0)

        @feed.Constraint(m.fs.time)
        def HCl_stoich_eqn(b, t):
            return (
                b.properties[t].conc_mol_comp["H"]
                == b.properties[t].conc_mol_comp["Cl"]
            )

        for condata in feed.HCl_stoich_eqn.values():
            set_scaling_factor(condata, 10)

<div class="alert alert-block alert-info">
<b>Engineering Insight:</b>
Since stronger acids are more expensive per unit volume and pose potential equipment and safety concerns (which would cost even more money to properly address), we'll assume that the pH of HCl feeds cannot drop below 0.
</div>

### Step 3.5 Set additional bounds on pH
For the reasons specified directly above, we should also bound the pH of other streams.

In [8]:
def optimize_pH(m):
    m.fs.leach_liquid_feed.properties[0].pH_phase["liquid"].setlb(0)
    m.fs.solex_rougher_scrub.mscontactor.aqueous[0.0, 1].pH_phase["liquid"].setub(4)

<div class="alert alert-block alert-info">
<b>Engineering Insight:</b>
If the pH in the rougher scrub goes above 4, aluminum hydroxide begins to precipitate out of solution.
<br> </br>
Reference: Zhang, W., Xie, X., Tong, X., Du, Y., Song, Q., & Feng, D. (2021). Study on the effect and mechanism of impurity aluminum on the solvent extraction of rare earth elements (Nd, Pr, La) by P204-P350 in chloride solution. Minerals, 11(1), 61.
</div>

### Step 3.5 Run the optimization
Now we'll simply create a function that combines all of the optimization-related functions into a single place.

In [9]:
def optimize_model(m, DEHPA_ub=10):
    create_objective(m)
    optimize_liquid_feed(m)
    optimize_organic_feed(m, DEHPA_ub=DEHPA_ub)
    optimize_acid_feed(m)
    optimize_pH(m)

Then we'll call the optimization function (`optimize_model`) to optimize the flowsheet and re-solve the system.

In [10]:
optimize_model(m1, DEHPA_ub=10)

solver = get_solver("ipopt_v2")
solver.options.constr_viol_tol = 1e-7
solver.options.max_iter = 300
results = solver.solve(m1, tee=True)

# Uncomment to display various system metrics like component recovery or leaching recovery
# uky.display_results(m1)

Ipopt 3.13.2: linear_solver="ma57"
max_iter=300
nlp_scaling_method="gradient-based"
tol=1e-06
constr_viol_tol=1e-07


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:


In [11]:
print(f"Total REE recovery is {value(m1.fs.overall_ree_recovery_percentage[0])} %")

Total REE recovery is 0.002814978843863314 %


## Step 4: Analyze the optimization results
Now that we have the system solving at 10% DEHPA, we'll need to update the tear guesses in the initialization routine so that we can try to solve the system at 20% DEHPA with a better starting point. 

### Step 4.1: Print the optimized conditions
First, let's display the values for the conditions optimized by the optimization function.

In [12]:
m1.fs.leach_liquid_feed.display()
m1.fs.rougher_org_make_up.display()
m1.fs.cleaner_org_make_up.display()
m1.fs.acid_feed1.display()
m1.fs.acid_feed2.display()
m1.fs.acid_feed3.display()

Block fs.leach_liquid_feed

  Variables:
    flow_vol : Size=1, Index=fs._time, ReferenceTo=fs.leach_liquid_feed.properties[:].component('flow_vol')
        Key : Lower : Value             : Upper : Fixed : Stale : Domain
        0.0 :   100 : 31946.61046104166 :  None : False : False :  Reals
    conc_mass_comp : Size=17, Index=fs._time*fs.leach_soln.component_list, ReferenceTo=fs.leach_liquid_feed.properties[:].component('conc_mass_comp')[...]
        Key           : Lower : Value              : Upper : Fixed : Stale : Domain
          (0.0, 'Al') : 1e-20 :              1e-10 :  None :  True :  True :  Reals
          (0.0, 'Ca') : 1e-20 :              1e-10 :  None :  True :  True :  Reals
          (0.0, 'Ce') : 1e-20 :              1e-10 :  None :  True :  True :  Reals
          (0.0, 'Cl') : 1e-20 :              1e-10 :  None :  True :  True :  Reals
          (0.0, 'Dy') : 1e-20 :              1e-10 :  None :  True :  True :  Reals
          (0.0, 'Fe') : 1e-20 :              1

<div class="alert alert-block alert-danger">
<b>WARNING:</b>
Uh oh... the flow rate of acid_feed1 is essentially being set to 0! Since this is suggesting that the system would perform better without the acid feed going into the solvent extraction rougher scrubbing unit, there must be some phenomenon that we're not modeling accurately. For now, we're just going to fix this flow rate to a reasonable flow rather than trying to optimize it.
</div>

### Step 4.2 Update the optimize_acid_feed function
Let's update the `optimize_acid_feed` function to fix the volumetric flow rate of acid_feed1 rather than trying to optimize it. It will be fixed at its default value specified in the flowsheet file (0.1 L/hr).

In [13]:
def optimize_acid_feed(m):
    # Unfix HCl feed flow rates and concentrations
    for feed in [m.fs.acid_feed1, m.fs.acid_feed2, m.fs.acid_feed3]:
        # Trying to optimize acid_feed1 will bring its flow to zero, so we'll keep it fixed
        if feed == m.fs.acid_feed1:
            continue
        else:
            feed.flow_vol.unfix()
        feed.conc_mass_comp[0, "H"].unfix()
        feed.conc_mass_comp[0, "Cl"].unfix()
        # Revisit how strong of an acid we can use
        feed.properties[0].pH_phase["liquid"].setlb(0)

        @feed.Constraint(m.fs.time)
        def HCl_stoich_eqn(b, t):
            return (
                b.properties[t].conc_mol_comp["H"]
                == b.properties[t].conc_mol_comp["Cl"]
            )

        for condata in feed.HCl_stoich_eqn.values():
            set_scaling_factor(condata, 10)

### Step 4.2 Re-run the optimization
Let's create a new model, `m2`, to represent this system, re-build the flowsheet, and then run optimization again.

In [14]:
m2 = uky.build()
uky.set_operating_conditions(m2, DEHPA_dosage=0.05)
uky.set_scaling(m2)
uky.initialize_system(m2)
uky.solve_system(m2, tee=False)
uky.fix_organic_recycle(m2)
uky.solve_system(m2, tee=False)
uky.add_result_expressions(m2)

Scaling fs.leach
Scaling fs.sl_sep1
Scaling fs.scrubber_HCl_leach_translator
Scaling fs.leach_mixer
Scaling fs.leach_liquid_feed
Scaling fs.leach_solid_feed
Scaling fs.leach_filter_cake
Scaling fs.leach_filter_cake_liquid
Scaling fs.rougher_org_make_up
Scaling fs.solex_rougher_load
Scaling fs.acid_feed1
Scaling fs.solex_rougher_scrub
Scaling fs.acid_feed2
Scaling fs.solex_rougher_strip
Scaling fs.rougher_sep
Scaling fs.rougher_mixer
Scaling fs.load_sep
Scaling fs.scrub_sep
Scaling fs.rougher_organic_purge
Scaling fs.solex_cleaner_load
Scaling fs.solex_cleaner_strip
Scaling fs.cleaner_org_make_up
Scaling fs.cleaner_mixer
Scaling fs.cleaner_sep
Scaling fs.cleaner_HCl_leach_translator
Scaling fs.leach_sx_mixer
Scaling fs.acid_feed3
Scaling fs.cleaner_organic_purge
Scaling fs.precipitator
Scaling fs.sl_sep2
Scaling fs.precip_sep
Scaling fs.precip_sx_mixer
Scaling fs.precip_purge
Scaling fs.roaster
Scaling fs.leaching_sol_feed_expanded
Scaling fs.leaching_liq_feed_expanded
Scaling fs.leachi

In [15]:
optimize_model(m2, DEHPA_ub=10)
uky.solve_system(m2, tee=False)

# Uncomment to display various system metrics like component recovery or leaching recovery
# uky.display_results(m2)

{'Problem': [{'Lower bound': None, 'Upper bound': -70.71927081062842, 'Number of objectives': 1, 'Number of constraints': nan, 'Number of variables': nan, 'Sense': 'minimize'}], 'Solver': [{'Status': 'ok', 'Termination condition': 'optimal', 'Termination message': 'TerminationCondition.convergenceCriteriaSatisfied'}]}

### Step 4.3 Reprint total recovery

In [16]:
print(f"Total REE recovery is {value(m2.fs.overall_ree_recovery_percentage[0])} %")

Total REE recovery is 11.703180792039845 %


## Step 5: Update the base conditions of the flowsheet
Now that we have the system successfully solving again at 10% DEHPA, we'll want to update the operating conditions to match the specs of the optimized system above. In order to do so, we'll need to update the initialization routine, which is currently hard-coding initial guesses based on the unoptimized system.

### Step 5.1 Print optimized operating conditions
Print the optimized variable values from `m2`.

In [17]:
m2.fs.leach_liquid_feed.display()
m2.fs.rougher_org_make_up.display()
m2.fs.cleaner_org_make_up.display()
m2.fs.acid_feed1.display()
m2.fs.acid_feed2.display()
m2.fs.acid_feed3.display()

Block fs.leach_liquid_feed

  Variables:
    flow_vol : Size=1, Index=fs._time, ReferenceTo=fs.leach_liquid_feed.properties[:].component('flow_vol')
        Key : Lower : Value              : Upper : Fixed : Stale : Domain
        0.0 :   100 : 100.00000087743352 :  None : False : False :  Reals
    conc_mass_comp : Size=17, Index=fs._time*fs.leach_soln.component_list, ReferenceTo=fs.leach_liquid_feed.properties[:].component('conc_mass_comp')[...]
        Key           : Lower : Value              : Upper : Fixed : Stale : Domain
          (0.0, 'Al') : 1e-20 :              1e-10 :  None :  True :  True :  Reals
          (0.0, 'Ca') : 1e-20 :              1e-10 :  None :  True :  True :  Reals
          (0.0, 'Ce') : 1e-20 :              1e-10 :  None :  True :  True :  Reals
          (0.0, 'Cl') : 1e-20 :              1e-10 :  None :  True :  True :  Reals
          (0.0, 'Dy') : 1e-20 :              1e-10 :  None :  True :  True :  Reals
          (0.0, 'Fe') : 1e-20 :             

### Step 5.2: Update the operating conditions
Let's create a new instance of the flowsheet, `m3`, and update the operating conditions to match those of the optimized case in `m2`.

In [18]:
m3 = uky.build()
uky.set_operating_conditions(m3, DEHPA_dosage=0.1)

m3.fs.leach_liquid_feed.flow_vol.fix(100)
m3.fs.leach_liquid_feed.conc_mass_comp[0, "H"].fix(194.5)
m3.fs.leach_liquid_feed.conc_mass_comp[0, "HSO4"].fix(17067.6)
m3.fs.leach_liquid_feed.conc_mass_comp[0, "SO4"].fix(888.8)
m3.fs.rougher_org_make_up.flow_vol.fix(40.03)
m3.fs.cleaner_org_make_up.flow_vol.fix(73.75)
m3.fs.acid_feed1.flow_vol.fix(0.1)
m3.fs.acid_feed1.conc_mass_comp[0, "H"].fix(1023)
m3.fs.acid_feed1.conc_mass_comp[0, "Cl"].fix(35980)
m3.fs.acid_feed2.flow_vol.fix(4.03)
m3.fs.acid_feed2.conc_mass_comp[0, "H"].fix(1008)
m3.fs.acid_feed2.conc_mass_comp[0, "Cl"].fix(35453)
m3.fs.acid_feed3.flow_vol.fix(4.19)
m3.fs.acid_feed3.conc_mass_comp[0, "H"].fix(1008)
m3.fs.acid_feed3.conc_mass_comp[0, "Cl"].fix(35453)

uky.set_scaling(m3)

Scaling fs.leach
Scaling fs.sl_sep1
Scaling fs.scrubber_HCl_leach_translator
Scaling fs.leach_mixer
Scaling fs.leach_liquid_feed
Scaling fs.leach_solid_feed
Scaling fs.leach_filter_cake
Scaling fs.leach_filter_cake_liquid
Scaling fs.rougher_org_make_up
Scaling fs.solex_rougher_load
Scaling fs.acid_feed1
Scaling fs.solex_rougher_scrub
Scaling fs.acid_feed2
Scaling fs.solex_rougher_strip
Scaling fs.rougher_sep
Scaling fs.rougher_mixer
Scaling fs.load_sep
Scaling fs.scrub_sep
Scaling fs.rougher_organic_purge
Scaling fs.solex_cleaner_load
Scaling fs.solex_cleaner_strip
Scaling fs.cleaner_org_make_up
Scaling fs.cleaner_mixer
Scaling fs.cleaner_sep
Scaling fs.cleaner_HCl_leach_translator
Scaling fs.leach_sx_mixer
Scaling fs.acid_feed3
Scaling fs.cleaner_organic_purge
Scaling fs.precipitator
Scaling fs.sl_sep2
Scaling fs.precip_sep
Scaling fs.precip_sx_mixer
Scaling fs.precip_purge
Scaling fs.roaster
Scaling fs.leaching_sol_feed_expanded
Scaling fs.leaching_liq_feed_expanded
Scaling fs.leachi

### Step 5.3: Update tear guesses
Here we recreate the entire initialization routine, but the only portion of this that is changing from the original implementation are the values for the tear guesses. Rather than just hard-coding new values, we will simply set the tear guesses equal to the value of the corresponding variable in `m2` to represent the new starting points.

In [19]:
def updated_initialization(m, ref=m2.fs):
    seq = SequentialDecomposition()
    seq.options.tear_method = "Direct"
    seq.options.iterLim = 1
    seq.options.tear_set = [
        m.fs.leaching_feed_mixture,
        m.fs.sx_rougher_load_aq_feed,
        m.fs.sx_rougher_mixed_org_recycle,
        m.fs.sx_cleaner_load_aq_feed,
        m.fs.sx_cleaner_mixed_org_recycle,
    ]

    tear_guesses1 = {
        "flow_vol": {0: value(ref.leach.liquid_inlet.flow_vol[0])},
        "conc_mass_comp": {
            (0, "Al"): value(ref.leach.liquid_inlet.conc_mass_comp[0, "Al"]),
            (0, "Ca"): value(ref.leach.liquid_inlet.conc_mass_comp[0, "Ca"]),
            (0, "Ce"): value(ref.leach.liquid_inlet.conc_mass_comp[0, "Ce"]),
            (0, "Cl"): value(ref.leach.liquid_inlet.conc_mass_comp[0, "Cl"]),
            (0, "Dy"): value(ref.leach.liquid_inlet.conc_mass_comp[0, "Dy"]),
            (0, "Fe"): value(ref.leach.liquid_inlet.conc_mass_comp[0, "Fe"]),
            (0, "Gd"): value(ref.leach.liquid_inlet.conc_mass_comp[0, "Gd"]),
            (0, "H"): value(ref.leach.liquid_inlet.conc_mass_comp[0, "H"]),
            (0, "H2O"): value(ref.leach.liquid_inlet.conc_mass_comp[0, "H2O"]),
            (0, "HSO4"): value(ref.leach.liquid_inlet.conc_mass_comp[0, "HSO4"]),
            (0, "La"): value(ref.leach.liquid_inlet.conc_mass_comp[0, "La"]),
            (0, "Nd"): value(ref.leach.liquid_inlet.conc_mass_comp[0, "Nd"]),
            (0, "Pr"): value(ref.leach.liquid_inlet.conc_mass_comp[0, "Pr"]),
            (0, "SO4"): value(ref.leach.liquid_inlet.conc_mass_comp[0, "SO4"]),
            (0, "Sc"): value(ref.leach.liquid_inlet.conc_mass_comp[0, "Sc"]),
            (0, "Sm"): value(ref.leach.liquid_inlet.conc_mass_comp[0, "Sm"]),
            (0, "Y"): value(ref.leach.liquid_inlet.conc_mass_comp[0, "Y"]),
        },
    }
    tear_guesses2 = {
        "flow_vol": {0: value(ref.solex_rougher_load.organic_inlet.flow_vol[0])},
        "conc_mass_comp": {
            (0, "Al_o"): value(
                ref.solex_rougher_load.organic_inlet.conc_mass_comp[0, "Al_o"]
            ),
            (0, "Ca_o"): value(
                ref.solex_rougher_load.organic_inlet.conc_mass_comp[0, "Ca_o"]
            ),
            (0, "Ce_o"): value(
                ref.solex_rougher_load.organic_inlet.conc_mass_comp[0, "Ce_o"]
            ),
            (0, "Dy_o"): value(
                ref.solex_rougher_load.organic_inlet.conc_mass_comp[0, "Dy_o"]
            ),
            (0, "Fe_o"): value(
                ref.solex_rougher_load.organic_inlet.conc_mass_comp[0, "Fe_o"]
            ),
            (0, "Gd_o"): value(
                ref.solex_rougher_load.organic_inlet.conc_mass_comp[0, "Gd_o"]
            ),
            (0, "La_o"): value(
                ref.solex_rougher_load.organic_inlet.conc_mass_comp[0, "La_o"]
            ),
            (0, "Nd_o"): value(
                ref.solex_rougher_load.organic_inlet.conc_mass_comp[0, "Nd_o"]
            ),
            (0, "Pr_o"): value(
                ref.solex_rougher_load.organic_inlet.conc_mass_comp[0, "Pr_o"]
            ),
            (0, "Sc_o"): value(
                ref.solex_rougher_load.organic_inlet.conc_mass_comp[0, "Sc_o"]
            ),
            (0, "Sm_o"): value(
                ref.solex_rougher_load.organic_inlet.conc_mass_comp[0, "Sm_o"]
            ),
            (0, "Y_o"): value(
                ref.solex_rougher_load.organic_inlet.conc_mass_comp[0, "Y_o"]
            ),
            (0, "DEHPA"): value(
                ref.solex_rougher_load.organic_inlet.conc_mass_comp[0, "DEHPA"]
            ),
            (0, "Kerosene"): value(
                ref.solex_rougher_load.organic_inlet.conc_mass_comp[0, "Kerosene"]
            ),
        },
    }
    tear_guesses3 = {
        "flow_vol": {0: value(ref.solex_rougher_load.aqueous_inlet.flow_vol[0])},
        "conc_mass_comp": {
            (0, "Al"): value(
                ref.solex_rougher_load.aqueous_inlet.conc_mass_comp[0, "Al"]
            ),
            (0, "Ca"): value(
                ref.solex_rougher_load.aqueous_inlet.conc_mass_comp[0, "Ca"]
            ),
            (0, "Ce"): value(
                ref.solex_rougher_load.aqueous_inlet.conc_mass_comp[0, "Ce"]
            ),
            (0, "Cl"): value(
                ref.solex_rougher_load.aqueous_inlet.conc_mass_comp[0, "Cl"]
            ),
            (0, "Dy"): value(
                ref.solex_rougher_load.aqueous_inlet.conc_mass_comp[0, "Dy"]
            ),
            (0, "Fe"): value(
                ref.solex_rougher_load.aqueous_inlet.conc_mass_comp[0, "Fe"]
            ),
            (0, "Gd"): value(
                ref.solex_rougher_load.aqueous_inlet.conc_mass_comp[0, "Gd"]
            ),
            (0, "H"): value(
                ref.solex_rougher_load.aqueous_inlet.conc_mass_comp[0, "H"]
            ),
            (0, "H2O"): value(
                ref.solex_rougher_load.aqueous_inlet.conc_mass_comp[0, "H2O"]
            ),
            (0, "HSO4"): value(
                ref.solex_rougher_load.aqueous_inlet.conc_mass_comp[0, "HSO4"]
            ),
            (0, "La"): value(
                ref.solex_rougher_load.aqueous_inlet.conc_mass_comp[0, "La"]
            ),
            (0, "Nd"): value(
                ref.solex_rougher_load.aqueous_inlet.conc_mass_comp[0, "Nd"]
            ),
            (0, "Pr"): value(
                ref.solex_rougher_load.aqueous_inlet.conc_mass_comp[0, "Pr"]
            ),
            (0, "SO4"): value(
                ref.solex_rougher_load.aqueous_inlet.conc_mass_comp[0, "SO4"]
            ),
            (0, "Sc"): value(
                ref.solex_rougher_load.aqueous_inlet.conc_mass_comp[0, "Sc"]
            ),
            (0, "Sm"): value(
                ref.solex_rougher_load.aqueous_inlet.conc_mass_comp[0, "Sm"]
            ),
            (0, "Y"): value(
                ref.solex_rougher_load.aqueous_inlet.conc_mass_comp[0, "Y"]
            ),
        },
    }
    tear_guesses4 = {
        "flow_vol": {0: value(ref.solex_cleaner_load.organic_inlet.flow_vol[0])},
        "conc_mass_comp": {
            (0, "Al_o"): value(
                ref.solex_cleaner_load.organic_inlet.conc_mass_comp[0, "Al_o"]
            ),
            (0, "Ca_o"): value(
                ref.solex_cleaner_load.organic_inlet.conc_mass_comp[0, "Ca_o"]
            ),
            (0, "Ce_o"): value(
                ref.solex_cleaner_load.organic_inlet.conc_mass_comp[0, "Ce_o"]
            ),
            (0, "Dy_o"): value(
                ref.solex_cleaner_load.organic_inlet.conc_mass_comp[0, "Dy_o"]
            ),
            (0, "Fe_o"): value(
                ref.solex_cleaner_load.organic_inlet.conc_mass_comp[0, "Fe_o"]
            ),
            (0, "Gd_o"): value(
                ref.solex_cleaner_load.organic_inlet.conc_mass_comp[0, "Gd_o"]
            ),
            (0, "La_o"): value(
                ref.solex_cleaner_load.organic_inlet.conc_mass_comp[0, "La_o"]
            ),
            (0, "Nd_o"): value(
                ref.solex_cleaner_load.organic_inlet.conc_mass_comp[0, "Nd_o"]
            ),
            (0, "Pr_o"): value(
                ref.solex_cleaner_load.organic_inlet.conc_mass_comp[0, "Pr_o"]
            ),
            (0, "Sc_o"): value(
                ref.solex_cleaner_load.organic_inlet.conc_mass_comp[0, "Sc_o"]
            ),
            (0, "Sm_o"): value(
                ref.solex_cleaner_load.organic_inlet.conc_mass_comp[0, "Sm_o"]
            ),
            (0, "Y_o"): value(
                ref.solex_cleaner_load.organic_inlet.conc_mass_comp[0, "Y_o"]
            ),
            (0, "DEHPA"): value(
                ref.solex_cleaner_load.organic_inlet.conc_mass_comp[0, "DEHPA"]
            ),
            (0, "Kerosene"): value(
                ref.solex_cleaner_load.organic_inlet.conc_mass_comp[0, "Kerosene"]
            ),
        },
    }
    tear_guesses5 = {
        "flow_vol": {0: value(ref.solex_cleaner_load.aqueous_inlet.flow_vol[0])},
        "conc_mass_comp": {
            (0, "Al"): value(
                ref.solex_cleaner_load.aqueous_inlet.conc_mass_comp[0, "Al"]
            ),
            (0, "Ca"): value(
                ref.solex_cleaner_load.aqueous_inlet.conc_mass_comp[0, "Ca"]
            ),
            (0, "Ce"): value(
                ref.solex_cleaner_load.aqueous_inlet.conc_mass_comp[0, "Ce"]
            ),
            (0, "Cl"): value(
                ref.solex_cleaner_load.aqueous_inlet.conc_mass_comp[0, "Cl"]
            ),
            (0, "Dy"): value(
                ref.solex_cleaner_load.aqueous_inlet.conc_mass_comp[0, "Dy"]
            ),
            (0, "Fe"): value(
                ref.solex_cleaner_load.aqueous_inlet.conc_mass_comp[0, "Fe"]
            ),
            (0, "Gd"): value(
                ref.solex_cleaner_load.aqueous_inlet.conc_mass_comp[0, "Gd"]
            ),
            (0, "H"): value(
                ref.solex_cleaner_load.aqueous_inlet.conc_mass_comp[0, "H"]
            ),
            (0, "H2O"): value(
                ref.solex_cleaner_load.aqueous_inlet.conc_mass_comp[0, "H2O"]
            ),
            (0, "La"): value(
                ref.solex_cleaner_load.aqueous_inlet.conc_mass_comp[0, "La"]
            ),
            (0, "Nd"): value(
                ref.solex_cleaner_load.aqueous_inlet.conc_mass_comp[0, "Nd"]
            ),
            (0, "Pr"): value(
                ref.solex_cleaner_load.aqueous_inlet.conc_mass_comp[0, "Pr"]
            ),
            (0, "Sc"): value(
                ref.solex_cleaner_load.aqueous_inlet.conc_mass_comp[0, "Sc"]
            ),
            (0, "Sm"): value(
                ref.solex_cleaner_load.aqueous_inlet.conc_mass_comp[0, "Sm"]
            ),
            (0, "Y"): value(
                ref.solex_cleaner_load.aqueous_inlet.conc_mass_comp[0, "Y"]
            ),
        },
    }

    # Pass the tear_guess to the SD tool
    seq.set_guesses_for(m.fs.leach.liquid_inlet, tear_guesses1)
    seq.set_guesses_for(m.fs.solex_rougher_load.organic_inlet, tear_guesses2)
    seq.set_guesses_for(m.fs.solex_rougher_load.aqueous_inlet, tear_guesses3)
    seq.set_guesses_for(m.fs.solex_cleaner_load.organic_inlet, tear_guesses4)
    seq.set_guesses_for(m.fs.solex_cleaner_load.aqueous_inlet, tear_guesses5)

    initializer_feed = FeedInitializer()
    feed_units = [
        m.fs.leach_liquid_feed,
        m.fs.leach_solid_feed,
        m.fs.rougher_org_make_up,
        m.fs.acid_feed1,
        m.fs.acid_feed2,
        m.fs.acid_feed3,
        m.fs.cleaner_org_make_up,
    ]

    initializer_product = ProductInitializer()
    product_units = [
        m.fs.leach_filter_cake,
        m.fs.leach_filter_cake_liquid,
        m.fs.cleaner_organic_purge,
        m.fs.rougher_organic_purge,
        m.fs.precip_purge,
    ]

    initializer_sep = SeparatorInitializer()
    sep_units = [
        m.fs.scrub_sep,
        m.fs.precip_sep,
    ]

    initializer_mix = MixerInitializer()
    mix_units = [
        m.fs.precip_sx_mixer,
    ]

    initializer_leach = LeachingTrainInitializer()
    leach_units = [
        m.fs.leach,
    ]

    initializer_sx = SolventExtractionInitializer()
    sx_units = [
        m.fs.solex_rougher_load,
        m.fs.solex_rougher_scrub,
        m.fs.solex_rougher_strip,
        m.fs.solex_cleaner_load,
        m.fs.solex_cleaner_strip,
    ]

    initializer_bt = BlockTriangularizationInitializer()

    def function(unit):
        if unit in feed_units:
            _log.info(f"Initializing {unit}")
            initializer_feed.initialize(unit)
        elif unit in product_units:
            _log.info(f"Initializing {unit}")
            initializer_product.initialize(unit)
        elif unit in sep_units:
            _log.info(f"Initializing {unit}")
            initializer_sep.initialize(unit)
        elif unit in mix_units:
            _log.info(f"Initializing {unit}")
            initializer_mix.initialize(unit)
        elif unit in leach_units:
            _log.info(f"Initializing {unit}")
            initializer_leach.initialize(unit)
        elif unit in sx_units:
            _log.info(f"Initializing {unit}")
            initializer_sx.initialize(unit)
        else:
            _log.info(f"Initializing {unit}")
            initializer_bt.initialize(unit)

    seq.run(m, function)

### Step 5.4: Solve the new system and print the recovery
We should expect the total REE recovery to match that of the optimized `m2` model.

In [20]:
updated_initialization(m3)
uky.solve_system(m3, tee=False)
uky.fix_organic_recycle(m3)
uky.solve_system(m3, tee=False)
uky.add_result_expressions(m3)

# Uncomment to display various system metrics like component recovery or leaching recovery
# uky.display_results(m3)

2026-06-25 10:53:18 [INFO] idaes.__main__: Initializing fs.leach_solid_feed
2026-06-25 10:53:18 [INFO] idaes.__main__: Initializing fs.leach_liquid_feed
2026-06-25 10:53:18 [INFO] idaes.__main__: Initializing fs.solex_rougher_load
2026-06-25 10:53:18 [INFO] idaes.init.fs.solex_rougher_load.mscontactor: Stream Initialization Completed.
2026-06-25 10:53:19 [INFO] idaes.init.fs.solex_rougher_load.mscontactor: Initialization Completed, optimal - <undefined>
2026-06-25 10:53:19 [INFO] idaes.__main__: Initializing fs.rougher_org_make_up
2026-06-25 10:53:19 [INFO] idaes.__main__: Initializing fs.acid_feed1
2026-06-25 10:53:19 [INFO] idaes.__main__: Initializing fs.acid_feed2
2026-06-25 10:53:19 [INFO] idaes.__main__: Initializing fs.solex_cleaner_load
2026-06-25 10:53:19 [INFO] idaes.init.fs.solex_cleaner_load.mscontactor: Stream Initialization Completed.
2026-06-25 10:53:19 [INFO] idaes.init.fs.solex_cleaner_load.mscontactor: Initialization Completed, optimal - <undefined>
2026-06-25 10:53:1

In [21]:
print(f"Total REE recovery is {value(m3.fs.overall_ree_recovery_percentage[0])} %")

Total REE recovery is 11.701773902385717 %


<div class="alert alert-block alert-success">
This matches the recovery we got from the optimized case in m2!
</div>

## Step 6: Run the optimization case for 20%
Now let's see if we can run an optimization on `m3` where the DEHPA dosage is 20%.

In [22]:
optimize_model(m3, DEHPA_ub=20)
uky.solve_system(m3, tee=False)

uky.display_results(m3)


Unit : fs.roaster                                                          Time: 0.0
------------------------------------------------------------------------------------
    Unit Performance


    Expressions: 

    Key                      : Value      : Units
    Product Al Mass Fraction :   0.027099 : dimensionless
    Product Fe Mass Fraction :   0.044784 : dimensionless
    Product Ca Mass Fraction :   0.052857 : dimensionless
    Product Sc Mass Fraction : 3.3135e-10 : dimensionless
     Product Y Mass Fraction :   0.014045 : dimensionless
    Product La Mass Fraction :   0.023887 : dimensionless
    Product Ce Mass Fraction :    0.54057 : dimensionless
    Product Pr Mass Fraction :  0.0038718 : dimensionless
    Product Nd Mass Fraction :    0.21854 : dimensionless
    Product Sm Mass Fraction :  0.0025543 : dimensionless
    Product Gd Mass Fraction :   0.054738 : dimensionless
    Product Dy Mass Fraction :   0.017049 : dimensionless

----------------------------------------

In [23]:
print(f"Total REE recovery is {value(m3.fs.overall_ree_recovery_percentage[0])} %")

Total REE recovery is 22.184783328242975 %


<div class="alert alert-block alert-success">
A significant improvement from the original 0.9% REE recovery!
</div>